# Análise regional — acidentes na Dutra (Piraí → Guaratinguetá)

Recorte do Sul do Vale do Paraíba, o trecho que de fato interessa: 14 municípios
cortados pela Dutra, de **Piraí (RJ)** até **Guaratinguetá (SP)**.

**Fonte principal deste recorte: PRF** — ela tem `municipio`, `latitude`/`longitude`,
`causa_acidente`, `condicao_metereologica` e `uso_solo`, que a ANTT não tem.

Objetivo: diagnóstico de causas e padrões (causa, clima, horário, uso do solo,
veículo, gravidade).

## Microtarefa 0 — conectar ao banco e recortar a região

Aqui a gente só abre o banco e separa as linhas da nossa região num DataFrame
chamado `df_regiao`. Ele vira a base de todas as próximas análises.

In [ ]:
# Bibliotecas que vamos usar
import sqlite3          # para conversar com o banco SQLite
import pandas as pd     # para trazer o resultado da query como tabela (DataFrame)

# Abre a conexão com o banco do projeto (mesmo caminho que você já usa no antt.ipynb)
conn = sqlite3.connect("/home/nathgobira/Documentos/Acidentes-Dutra/data/processed/acidentes.db")

# Os 14 municípios cortados pela Dutra no nosso recorte, na ordem da estrada (Rio -> SP).
# Definimos UMA vez aqui e reaproveitamos em todas as análises seguintes.
MUNICIPIOS_REGIAO = [
    # Rio de Janeiro
    "PIRAI", "PINHEIRAL", "VOLTA REDONDA", "BARRA MANSA",
    "PORTO REAL", "RESENDE", "ITATIAIA",
    # São Paulo
    "QUELUZ", "LAVRINHAS", "CRUZEIRO", "CACHOEIRA PAULISTA",
    "CANAS", "LORENA", "GUARATINGUETA",
]

# Monta um "?" para cada município: vira "?,?,?,..." (14 pontos de interrogação).
# É o jeito SEGURO de passar uma lista para um SELECT ... IN (...):
# evita erro de aspas e protege contra SQL injection.
placeholders = ",".join(["?"] * len(MUNICIPIOS_REGIAO))

# A query: todas as colunas da PRF, só das linhas cujo município está na lista.
query = f"SELECT * FROM acidentes_prf WHERE municipio IN ({placeholders})"

# Executa a query. 'params' preenche os "?" com os nomes da lista, na ordem.
df_regiao = pd.read_sql(query, conn, params=MUNICIPIOS_REGIAO)

print(f"Linhas no recorte regional: {len(df_regiao)}")

In [ ]:
# --- Verificação ---
# 1) O total tem que bater com os 3001 que validamos direto no SQL.
print("Total de acidentes na região:", len(df_regiao))

# 2) Quantos acidentes por município? Confere se aparecem só os 14 esperados
#    (nenhum município de fora, como GUARULHOS ou REGISTRO).
print("\nAcidentes por município:")
print(df_regiao["municipio"].value_counts())

**Próxima microtarefa (4 do plano):** contagem por `causa_acidente` na região + gráfico.
Só depois que você rodar as células acima e confirmar que deu 3001.